# Election Polls + Results → Long Modeling Dataset

**Goal (this stage):** keep as much data as possible. Output is **one row per individual poll**, with the race's eventual result attached. No collapsing yet — that comes later.

So the table is one-to-many: many poll rows per race, each carrying who won and the final result.

**Scope:** Senate, House, Governor · 2000–present · U.S.

**Sources (both public, free):**
- **Polls:** FiveThirtyEight raw poll files (`*_polls.csv`) — current + historical
- **Results (who won):** FiveThirtyEight `election-results` repo — has a per-candidate `winner` flag

Run top to bottom. First run downloads into a local `data/` folder and caches it.

In [ ]:
import os, io, re, unicodedata
import pandas as pd
import numpy as np
import requests

pd.set_option('display.max_columns', 80)
DATA_DIR = 'data'
os.makedirs(DATA_DIR, exist_ok=True)

MIN_YEAR = 2000          # results go back to 1998; polling thins out pre-2000
OFFICES  = ['Senate', 'House', 'Governor']
print('pandas', pd.__version__)

## 1. Download with fallback mirrors

Both current and **historical** poll files are pulled (history lives in the `*_historical.csv` files). Result files come from GitHub with a CDN fallback. Everything is cached in `data/`.

If a poll URL ever 404s (538 occasionally relocates them), paste a working URL into the matching list below.

In [ ]:
def fetch(name, urls, force=False):
    """Download first working URL to data/<name>, with cache. Returns local path or None."""
    path = os.path.join(DATA_DIR, name)
    if os.path.exists(path) and not force:
        print(f'cached  {name}  ({os.path.getsize(path)//1024} KB)')
        return path
    last = None
    for u in urls:
        try:
            r = requests.get(u, timeout=60, headers={'User-Agent': 'Mozilla/5.0'})
            r.raise_for_status()
            head = r.content[:200].lstrip().lower()
            if head.startswith(b'<!doctype') or head.startswith(b'<html'):
                raise ValueError('got HTML, not CSV')
            with open(path, 'wb') as f:
                f.write(r.content)
            print(f'OK      {name}  ({len(r.content)//1024} KB)  <- {u}')
            return path
        except Exception as e:
            last = f'{type(e).__name__}: {e}'
            print(f'  skip  {u}  ({last})')
    print(f'  !! all mirrors failed for {name}: {last}')
    return None

P = 'https://projects.fivethirtyeight.com/polls-page/data/'

# Each office -> list of poll files (current + historical). Missing files are skipped gracefully.
poll_files = {
    'Senate':   [fetch('senate_polls.csv',            [P + 'senate_polls.csv']),
                 fetch('senate_polls_historical.csv',  [P + 'senate_polls_historical.csv'])],
    'House':    [fetch('house_polls.csv',             [P + 'house_polls.csv']),
                 fetch('house_polls_historical.csv',   [P + 'house_polls_historical.csv'])],
    'Governor': [fetch('governor_polls.csv',          [P + 'governor_polls.csv']),
                 fetch('governor_polls_historical.csv',[P + 'governor_polls_historical.csv'])],
}
poll_files = {o: [p for p in lst if p] for o, lst in poll_files.items()}

def res_urls(fn):
    return [
        f'https://raw.githubusercontent.com/fivethirtyeight/election-results/main/{fn}',
        f'https://cdn.jsdelivr.net/gh/fivethirtyeight/election-results@main/{fn}',
    ]
result_files = {
    'Senate':   fetch('res_senate.csv',   res_urls('election_results_senate.csv')),
    'House':    fetch('res_house.csv',    res_urls('election_results_house.csv')),
    'Governor': fetch('res_governor.csv', res_urls('election_results_gubernatorial.csv')),
}

## 2. Inspect raw columns

Schemas drift over time. Run once and eyeball the columns; if the standardizing step below references a column that's missing, adjust the `pick()` calls.

In [ ]:
for office in OFFICES:
    print(f'=== {office} POLL FILES ===')
    for f in poll_files.get(office, []):
        df = pd.read_csv(f, nrows=3, low_memory=False)
        print(f'  {os.path.basename(f)}  ({df.shape[1]} cols): {list(df.columns)}')
    if result_files.get(office):
        r = pd.read_csv(result_files[office], nrows=3, low_memory=False)
        print(f'  RESULTS ({r.shape[1]} cols): {list(r.columns)}')
    print()

## 3. Helpers + load polls (LONG: one row per poll-candidate)

We keep every poll row and a generous set of columns (pollster, dates, sample size, rating, methodology, etc.) so nothing is thrown away. Join keys are normalized but the original columns are preserved.

In [ ]:
def norm_name(s):
    """Normalize a candidate name for joining: strip accents, lowercase, last name + first initial."""
    if pd.isna(s):
        return None
    s = unicodedata.normalize('NFKD', str(s)).encode('ascii', 'ignore').decode().lower()
    s = re.sub(r'\b(jr|sr|ii|iii|iv)\b', '', s)
    s = re.sub(r'[^a-z\s]', ' ', s)
    parts = [w for w in s.split() if w]
    if not parts:
        return None
    last = parts[-1]
    fi = parts[0][0] if parts[0] != last else ''
    return f'{last} {fi}'.strip()

def norm_party(p):
    if pd.isna(p):
        return 'OTH'
    p = str(p).strip().upper()
    if p.startswith('DEM'): return 'DEM'
    if p.startswith('REP'): return 'REP'
    return 'OTH'

def pick(df, *cands):
    lower = {c.lower(): c for c in df.columns}
    for c in cands:
        if c.lower() in lower:
            return lower[c.lower()]
    return None

In [ ]:
def load_polls(paths, office):
    frames = []
    for path in paths:
        df = pd.read_csv(path, low_memory=False)
        c_year  = pick(df, 'cycle', 'election_year', 'year')
        c_state = pick(df, 'state')
        c_cand  = pick(df, 'candidate_name', 'answer', 'candidate')
        c_party = pick(df, 'party')
        c_pct   = pick(df, 'pct')
        c_date  = pick(df, 'end_date', 'enddate')
        # standardized join/feature columns
        df = df.rename(columns={
            c_year: 'year', c_state: 'state', c_cand: 'candidate',
            c_party: 'poll_party', c_pct: 'pct', c_date: 'end_date',
        })
        df['office'] = office
        df['year'] = pd.to_numeric(df['year'], errors='coerce')
        df['pct']  = pd.to_numeric(df['pct'], errors='coerce')
        df['end_date'] = pd.to_datetime(df['end_date'], errors='coerce')
        df['cand_key']  = df['candidate'].map(norm_name)
        df['party_std'] = df['poll_party'].map(norm_party)
        df = df.dropna(subset=['year', 'pct', 'cand_key'])
        df['year'] = df['year'].astype(int)
        frames.append(df)
    out = pd.concat(frames, ignore_index=True)
    # drop exact-duplicate poll rows that overlap between current & historical files
    dedup_cols = [c for c in ['poll_id','question_id','candidate','pct','end_date'] if c in out.columns]
    if dedup_cols:
        out = out.drop_duplicates(subset=dedup_cols)
    return out

polls = pd.concat([load_polls(poll_files[o], o) for o in OFFICES if poll_files.get(o)],
                  ignore_index=True)
polls = polls[polls['year'] >= MIN_YEAR]
print('total poll rows (long):', len(polls))
print('by office:\n', polls.groupby('office').size())
polls[['year','state','office','candidate','party_std','pct','end_date','cand_key']].head()

## 4. Load results (the outcome per candidate per race)

In [ ]:
def load_results(path, office):
    df = pd.read_csv(path, low_memory=False)
    sc = pick(df, 'stage')
    if sc:
        df = df[df[sc].astype(str).str.lower().str.contains('general', na=False)]
    c_year  = pick(df, 'cycle', 'year')
    c_state = pick(df, 'state_abbrev', 'state')
    c_cand  = pick(df, 'candidate_name', 'candidate')
    c_party = pick(df, 'party', 'ballot_party')
    c_pct   = pick(df, 'percent', 'pct')
    c_win   = pick(df, 'winner')
    out = pd.DataFrame({
        'year':      pd.to_numeric(df[c_year], errors='coerce'),
        'state':     df[c_state].astype(str).str.strip(),
        'office':    office,
        'res_candidate': df[c_cand],
        'res_party':     df[c_party].map(norm_party),
        'vote_pct':      pd.to_numeric(df[c_pct], errors='coerce'),
        'won':           df[c_win],
    })
    out['won'] = (out['won'].astype(str).str.lower()
                  .isin(['true','1','t','yes','y'])).astype(int)
    out['cand_key'] = out['res_candidate'].map(norm_name)
    out = out.dropna(subset=['year', 'cand_key'])
    out['year'] = out['year'].astype(int)
    # winning vote share in each race (for race-level margin features later)
    out['race_id'] = out['year'].astype(str)+'_'+out['state']+'_'+out['office']
    out['race_winning_pct'] = out.groupby('race_id')['vote_pct'].transform('max')
    return out

results = pd.concat([load_results(result_files[o], o) for o in OFFICES if result_files.get(o)],
                    ignore_index=True)
results = results[results['year'] >= MIN_YEAR]
res_join = results[['year','state','office','cand_key',
                    'won','vote_pct','res_party','res_candidate','race_winning_pct']]\
                  .drop_duplicates(['year','state','office','cand_key'])
print('result rows:', len(results), ' | unique candidate-races:', len(res_join))
res_join.head()

## 5. Join: attach each race's result onto every poll row

Left join keeps **all** poll rows. `_merge`/`has_result` tells you which polls matched a result, so you keep maximum data and can filter later for modeling.

In [ ]:
long = polls.merge(res_join, on=['year','state','office','cand_key'],
                   how='left', indicator=True)
long['has_result'] = (long['_merge'] == 'both').astype(int)
long = long.drop(columns='_merge')
long['race_id'] = long['year'].astype(str)+'_'+long['state']+'_'+long['office']

print('long rows:', len(long))
print('match rate by office:')
print(long.groupby('office')['has_result'].mean().round(3))
print('\npolls with a known winner:', long['won'].notna().sum())
long[['year','state','office','candidate','pct','end_date',
      'res_candidate','won','vote_pct','has_result']].head(12)

## 6. Save the long dataset

This is the master file: one row per poll, result attached. Collapse to one-row-per-race later when you're ready to model.

In [ ]:
OUT = 'polls_long_with_results.csv'
long.to_csv(OUT, index=False)
print('saved ->', os.path.abspath(OUT))
print('shape :', long.shape)
print('years :', long['year'].min(), '-', long['year'].max())
print('cols  :', list(long.columns))

## Notes

- **Long format on purpose:** every poll is a row, so you retain pollster, dates, sample size, methodology, ratings — aggregate however you like later (e.g. recency-weighted average, last-21-days, pollster-rating weighting).
- **`has_result == 0`** = a poll whose candidate didn't match a result row (usually a name-matching miss, a primary, or a dropped-out candidate). Keep them for now; filter to `has_result == 1` when training.
- **House** matches worst (sparse district polling); Senate/Governor match best.
- **Collapsing later:** group by `race_id`, build features per candidate, then pivot to one row per race. Don't feed `vote_pct` / `race_winning_pct` into the model as inputs — they're outcomes.
- If match rate looks low, inspect `long[long.has_result==0]` and refine `norm_name`.